In [39]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-08-12
Last modified on 2024-08-12
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener reglas de detección asociadas a una lista de TTP's así como la generación de un archivo .md con la información complementaria a la solicitud.

'''

"\nCreated on 2024-08-12\nLast modified on 2024-08-12\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener reglas de detección asociadas a una lista de TTP's así como la generación de un archivo .md con la información complementaria a la solicitud.\n\n"

**Requerimientos**

In [40]:
import os
import pandas as pd
import shutil
# import requests
from stix2 import Filter, MemoryStore
# import stix2

**IMPORTANTE - REQUERIMIENTOS PREVIOS**

- Requiere haber ejecutado previamente dentro de este mismo módulo **get_rules_and_classify_by_ttp** el notebook **get_rules_and_classify_by_ttp.ipynb** y disponer de la carpeta "outputs". 

- Imprescindible: ejecución del módulo **mitre_relationships** con los outputs correspondientes a stix2 (enterprise, ics, mobile...information, relations...techniques_tactics, techniques_groups, techniques_platforms, techniques_datasources, techniques_software...). Para la generación del archivo .md informativo.

**Parámetros**

In [41]:
ttp_list = ['T1114', 'T1048', 'T1486']
mitre_domain = 'enterprise' # enterprise, mobile, ics
optional_folder = 'Data-Xfil-Ext-Critical'

In [42]:
# Parámetros NO modificables
rules_path = os.path.join(os.getcwd(), 'outputs', mitre_domain)
output_path = os.path.join(os.getcwd(), 'query_outputs')
source_folders = [folder for folder in os.listdir(rules_path) if os.path.isdir(os.path.join(rules_path, folder))]

##### Funciones

In [43]:
def get_rules_df(folder_source, list_ttp, path_rules):
    '''
     Función cuyo cometido es crear un dataframe que contenga el id de la ttp, nombre de la regla, origen y ruta.
    '''
    # Crear un DataFrame vacío con las columnas 'ttp', 'rule', 'source', 'path'
    df = pd.DataFrame(columns=['technique ID', 'rule', 'source', 'path'])
    # Iterar sobre las carpetas de origen
    for source in folder_source:
        # Iterar sobre la lista de TTP
        for ttp in list_ttp:
            # Construir la ruta a las reglas
            rules = os.path.join(path_rules, source, ttp)
            # Comprobar si la ruta existe
            if os.path.exists(rules):
                # Listar los archivos en la carpeta de reglas
                rule_list = os.listdir(rules)
                # Crear las rutas completas para cada regla
                full_paths = [os.path.join(rules, rule) for rule in rule_list]
                # Crear un DataFrame temporal con la información actual
                temp_df = pd.DataFrame({
                    'technique ID': [ttp] * len(rule_list),
                    'rule': rule_list,
                    'source': [source] * len(rule_list),
                    'path': full_paths
                })
                # Concatenar el DataFrame temporal con el principal
                df = pd.concat([df, temp_df], ignore_index=True)

    # Ordenar el DataFrame por el campo 'ttp' en orden descendente y resetear el índice
    return df.sort_values(by='technique ID', ascending=False).reset_index(drop=True)

In [44]:
def copy_files(df, path_output, optional_folder):
    '''
     Función encargada de rercorrer el dataframe y copiar los archivos a una nueva estructura de carpetas.
    '''
    # Recorrer cada fila del DataFrame
    for _, row in df.iterrows():
        # Obtener el ttp, rule y la ruta completa del archivo
        ttp = row['technique ID']
        rule = row['rule']
        source_path = row['path']

        # Crear la ruta destino en la estructura output/[technique ID]/[rule]
        destination_dir = os.path.join(path_output, optional_folder, ttp)
        destination_path = os.path.join(destination_dir, rule)

        # Crear las carpetas si no existen
        os.makedirs(destination_dir, exist_ok=True)

        # Copiar el archivo al destino
        shutil.copy(source_path, destination_path)
        print(f"Archivo copiado: {source_path} -> {destination_path}")

In [45]:
def get_data_from_techniques(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_techniques.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\techniques',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [46]:
def get_data_from_relation_techniques_tactics(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_tactics_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_tactics',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [47]:
def get_data_from_relation_techniques_software(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_software_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_software',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [48]:
def get_data_from_relation_techniques_platforms(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_platforms_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_platforms',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df


In [49]:
def get_data_from_relation_techniques_groups(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_groups_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_groups',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [50]:
def get_data_from_relation_techniques_datasources(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_datasources_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_datasources',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [51]:
def format_cols_for_md(column):
    '''
    Función para aplicar los reemplazos
    '''
    if column.dtype == "object":  # Solo aplica a columnas de tipo string
        column = column.str.replace("'", '', regex=False)
        column = column.str.replace("[", '[[', regex=False)
        column = column.str.replace("]", ']]', regex=False)
        column = column.str.replace(", ", ']] [[', regex=False)
    return column

#### **Ejecución**

##### **1. Obtención de la lista de reglas de detección por ttp asociada**

In [52]:
results_df = get_rules_df(source_folders, ttp_list, rules_path)
results_df.head(3)

,technique ID,rule,source,path
0,T1486,proc_creation_win_gpg4win_portable_execution.yml,Sigma HQ 1,c:\Users\jelopez\Documents\CyberProof\python\d...
1,T1486,ActifioGo.yaml,Mappings,c:\Users\jelopez\Documents\CyberProof\python\d...
2,T1486,proc_creation_win_reg_bitlocker.yml,Sigma HQ 1,c:\Users\jelopez\Documents\CyberProof\python\d...


##### **2. Generación de la estructura de carpetas por ttp con las reglas de detección**

In [53]:
copy_files(results_df, output_path, optional_folder)

Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\Sigma HQ 1\T1486\proc_creation_win_gpg4win_portable_execution.yml -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Data-Xfil-Ext-Critical\T1486\proc_creation_win_gpg4win_portable_execution.yml
Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\Mappings\T1486\ActifioGo.yaml -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Data-Xfil-Ext-Critical\T1486\ActifioGo.yaml
Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\Sigma HQ 1\T1486\proc_creation_win_reg_bitlocker.yml -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Data-Xfil-Ext-Critical\T1486\proc_creation_win_reg_bitlocker.yml
Archivo

##### **3. Obtención de la información relativa a la consulta**

In [54]:
techniques_df = get_data_from_techniques(mitre_domain)
# Filtramos la información de técnicas asociadas 
techniques_df = techniques_df[techniques_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_df = techniques_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
# Formateo para la posterior unión
techniques_df['technique_ID'] = techniques_df['technique_ID'].apply(lambda x: f'[[{x}]]')
techniques_df['technique'] = techniques_df['technique'].apply(lambda x: f'[[{x}]]')
techniques_df

,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked,matrix_domains
0,[[T1486]],[[Data Encrypted for Impact]],https://attack.mitre.org/techniques/T1486,Adversaries may encrypt data on target systems...,False,False,['enterprise-attack']
1,[[T1114]],[[Email Collection]],https://attack.mitre.org/techniques/T1114,Adversaries may target user email to collect s...,False,False,['enterprise-attack']
2,[[T1048]],[[Exfiltration Over Alternative Protocol]],https://attack.mitre.org/techniques/T1048,Adversaries may steal data by exfiltrating it ...,False,False,['enterprise-attack']


In [55]:
techniques_tactics_df = get_data_from_relation_techniques_tactics(mitre_domain)
techniques_tactics_df = techniques_tactics_df[techniques_tactics_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_tactics_df = techniques_tactics_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
for col in techniques_tactics_df.columns:
    techniques_tactics_df[col] = format_cols_for_md(techniques_tactics_df[col])
techniques_tactics_df

,technique_ID,technique,tactic_ID,tactic
0,T1486,[[Data Encrypted for Impact]],[[TA0040]],[[Impact]]
1,T1114,[[Email Collection]],[[TA0009]],[[Collection]]
2,T1048,[[Exfiltration Over Alternative Protocol]],[[TA0010]],[[Exfiltration]]


In [56]:
techniques_software_df = get_data_from_relation_techniques_software(mitre_domain)
techniques_software_df = techniques_software_df[techniques_software_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_software_df = techniques_software_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
for col in techniques_software_df.columns:
    techniques_software_df[col] = format_cols_for_md(techniques_software_df[col])
techniques_software_df

,technique_ID,technique,software_ID,software
0,T1486,[[Data Encrypted for Impact]],[[S0595]] [[S1053]] [[S1111]] [[S0583]] [[S057...,[[BitPaymer]] [[Akira]] [[RobbinHood]] [[Thief...
1,T1114,[[Email Collection]],[[S0367]],[[Emotet]]
2,T1048,[[Exfiltration Over Alternative Protocol]],[[S0482]] [[S0677]] [[S0203]] [[S0641]] [[S042...,[[Hydraq]] [[AADInternals]] [[Chaes]] [[Bundlo...


In [57]:
techniques_platforms_df = get_data_from_relation_techniques_platforms(mitre_domain)
techniques_platforms_df = techniques_platforms_df[techniques_platforms_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_platforms_df = techniques_platforms_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
for col in techniques_platforms_df.columns:
    techniques_platforms_df[col] = format_cols_for_md(techniques_platforms_df[col])
techniques_platforms_df

,technique_ID,technique,platform
0,T1486,[[Data Encrypted for Impact]],[[Windows]] [[Linux]] [[macOS]] [[IaaS]]
1,T1114,[[Email Collection]],[[Windows]] [[Linux]] [[Google Workspace]] [[m...
2,T1048,[[Exfiltration Over Alternative Protocol]],[[Windows]] [[IaaS]] [[Linux]] [[SaaS]] [[Goog...


In [58]:
techniques_groups_df = get_data_from_relation_techniques_groups(mitre_domain)
techniques_groups_df = techniques_groups_df[techniques_groups_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_groups_df = techniques_groups_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
for col in techniques_groups_df.columns:
    techniques_groups_df[col] = format_cols_for_md(techniques_groups_df[col])
techniques_groups_df

,technique_ID,technique,group_ID,group
0,T1486,[[Data Encrypted for Impact]],[[G1015]] [[G0096]] [[G0061]] [[G0059]] [[G102...,[[Akira]] [[APT41]] [[FIN8]] [[Scattered Spide...
1,T1114,[[Email Collection]],[[G0122]] [[G0059]],[[Silent Librarian]] [[Magic Hound]]
2,T1048,[[Exfiltration Over Alternative Protocol]],[[G0139]],[[TeamTNT]]


In [59]:
techniques_datasources_df = get_data_from_relation_techniques_datasources(mitre_domain)
techniques_datasources_df = techniques_datasources_df[techniques_datasources_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
techniques_datasources_df = techniques_datasources_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
for col in techniques_datasources_df.columns:
    techniques_datasources_df[col] = format_cols_for_md(techniques_datasources_df[col])
techniques_datasources_df

,technique_ID,technique,data_source_ID,data_source
0,T1486,[[Data Encrypted for Impact]],[[DS0017]] [[DS0009]] [[DS0033]] [[DS0010]] [[...,[[File]] [[Command]] [[Cloud Storage]] [[Netwo...
1,T1114,[[Email Collection]],[[DS0029]] [[DS0017]] [[DS0028]] [[DS0015]] [[...,[[Logon Session]] [[File]] [[Command]] [[Netwo...
2,T1048,[[Exfiltration Over Alternative Protocol]],[[DS0029]] [[DS0017]] [[DS0015]] [[DS0010]] [[...,[[File]] [[Command]] [[Cloud Storage]] [[Netwo...


##### **4. Unión de las tablas informativas y formateos previos a la generación del .md**

In [60]:
info_df = pd.merge(techniques_tactics_df, techniques_software_df, on=['technique_ID', 'technique'], how='left')
info_df = pd.merge(info_df, techniques_platforms_df, on=['technique_ID', 'technique'], how='left')
info_df = pd.merge(info_df, techniques_groups_df, on=['technique_ID', 'technique'], how='left')
info_df = pd.merge(info_df, techniques_datasources_df, on=['technique_ID', 'technique'], how='left')
info_df['technique_ID'] = info_df['technique_ID'].apply(lambda x: f'[[{x}]]')
info_df = pd.merge(info_df, techniques_df, on=['technique_ID', 'technique'], how='left')
info_df.columns = info_df.columns.str.replace('_', ' ', regex=False)
info_df['matrix domains'] = info_df['matrix domains'].apply(lambda x: x[2:-2])
info_df

,technique ID,technique,tactic ID,tactic,software ID,software,platform,group ID,group,data source ID,data source,technique url,technique description,technique deprecated,technique revoked,matrix domains
0,[[T1486]],[[Data Encrypted for Impact]],[[TA0040]],[[Impact]],[[S0595]] [[S1053]] [[S1111]] [[S0583]] [[S057...,[[BitPaymer]] [[Akira]] [[RobbinHood]] [[Thief...,[[Windows]] [[Linux]] [[macOS]] [[IaaS]],[[G1015]] [[G0096]] [[G0061]] [[G0059]] [[G102...,[[Akira]] [[APT41]] [[FIN8]] [[Scattered Spide...,[[DS0017]] [[DS0009]] [[DS0033]] [[DS0010]] [[...,[[File]] [[Command]] [[Cloud Storage]] [[Netwo...,https://attack.mitre.org/techniques/T1486,Adversaries may encrypt data on target systems...,False,False,enterprise-attack
1,[[T1114]],[[Email Collection]],[[TA0009]],[[Collection]],[[S0367]],[[Emotet]],[[Windows]] [[Linux]] [[Google Workspace]] [[m...,[[G0122]] [[G0059]],[[Silent Librarian]] [[Magic Hound]],[[DS0029]] [[DS0017]] [[DS0028]] [[DS0015]] [[...,[[Logon Session]] [[File]] [[Command]] [[Netwo...,https://attack.mitre.org/techniques/T1114,Adversaries may target user email to collect s...,False,False,enterprise-attack
2,[[T1048]],[[Exfiltration Over Alternative Protocol]],[[TA0010]],[[Exfiltration]],[[S0482]] [[S0677]] [[S0203]] [[S0641]] [[S042...,[[Hydraq]] [[AADInternals]] [[Chaes]] [[Bundlo...,[[Windows]] [[IaaS]] [[Linux]] [[SaaS]] [[Goog...,[[G0139]],[[TeamTNT]],[[DS0029]] [[DS0017]] [[DS0015]] [[DS0010]] [[...,[[File]] [[Command]] [[Cloud Storage]] [[Netwo...,https://attack.mitre.org/techniques/T1048,Adversaries may steal data by exfiltrating it ...,False,False,enterprise-attack


##### **5. Formateo de la tabla resumen de reglas de detección por técnica**

In [61]:
results_df = results_df[['technique ID', 'source', 'rule']]
results_df.head(3)

,technique ID,source,rule
0,T1486,Sigma HQ 1,proc_creation_win_gpg4win_portable_execution.yml
1,T1486,Mappings,ActifioGo.yaml
2,T1486,Sigma HQ 1,proc_creation_win_reg_bitlocker.yml


##### **6. Guardado del archivo markdown con la información relativa a la query**

In [62]:
def generate_markdown(df_techniques, df_additional, folder_path, optional_folder=''):
    # Asegúrate de que la carpeta exista
    os.makedirs(folder_path, exist_ok=True)
    
    # Construir la ruta completa del archivo
    filename = f'query_information_{optional_folder}.md'  # Nombre fijo para el archivo Markdown
    file_path = os.path.join(folder_path, filename)
    
    with open(file_path, 'w') as file:
        # Agregar la tabla adicional al inicio del archivo
        file.write("### Resumen reglas de detección:\n\n")
        
        # Convertir df_additional a formato tabla Markdown
        file.write("| technique ID | source | rule |\n")
        file.write("| ------------ | ------ | ---- |\n")
        for _, row in df_additional.iterrows():
            file.write(f"| {row['technique ID']} | {row['source']} | {row['rule']} |\n")
        file.write("\n\n")
        
        # Agregar el resto del contenido del DataFrame df_techniques
        for index, row in df_techniques.iterrows():
            # Título de la técnica
            file.write(f"### {row['technique ID'].strip('[]')}\n\n")
            
            # Información de la técnica
            file.write(f"**technique ID**\n{row['technique ID']}\n\n")
            file.write(f"**technique**\n{row['technique']}\n\n")
            file.write(f"**technique url**\n{row['technique url']}\n\n")
            file.write(f"**technique description**\n{row['technique description']}\n\n")
            file.write(f"**technique deprecated**\n{row['technique deprecated']}\n\n")
            file.write(f"**technique revoked**\n{row['technique revoked']}\n\n")
            file.write(f"**matrix domains**\n{row['matrix domains']}\n\n")
            file.write(f"**tactic ID**\n{row['tactic ID']}\n\n")
            file.write(f"**tactic**\n{row['tactic']}\n\n")
            file.write(f"**software ID**\n{row['software ID']}\n\n")
            file.write(f"**software**\n{row['software']}\n\n")
            file.write(f"**platform**\n{row['platform']}\n\n")
            file.write(f"**group ID**\n{row['group ID']}\n\n")
            file.write(f"**group**\n{row['group']}\n\n")
            file.write(f"**data source ID**\n{row['data source ID']}\n\n")
            file.write(f"**data source**\n{row['data source']}\n\n")

            # Añadir una línea en blanco para separar secciones
            file.write("\n\n")

    print(f"Archivo '{filename}' creado en la carpeta '{folder_path}' con éxito.")


In [63]:
generate_markdown(info_df, results_df, os.path.join(output_path,optional_folder), optional_folder)

Archivo 'query_information_Data-Xfil-Ext-Critical.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Data-Xfil-Ext-Critical' con éxito.
